 # 2.02 — Regression Benchmarking: 3-Hour Horizon



 I want to pick a regression model before committing to training across all 6

 horizons. The 3-hour horizon is a good benchmark target — it's short enough

 that availability lags still carry real signal, but long enough that weather

 and demand patterns start to matter too.



 Three models, walk-forward CV with tuned hyperparameters:

 - **OLS / Linear Regression** — floor benchmark. No regularization, no tuning.

   statsmodels gives the full inference table: coefficients, t-stats, p-values,

   confidence intervals.

 - **Ridge** — L2 regularization. Optuna searches alpha on the full dataset.

 - **LGBMRegressor (LightGBM)** — expected winner. Optuna tunes 5 hyperparameters;

   early stopping handles n_estimators inside each trial. I picked LightGBM over

   XGBoost for training time: my GPU is AMD (no CUDA), so everything runs on CPU,

   and LightGBM's histogram-native, leaf-wise design is 3–5× faster on CPU at this

   row count for ~the same accuracy on tabular data.



 All three are evaluated on the same 5-fold walk-forward CV so the RMSE/RAE

 comparison is apples-to-apples. Hyperparameter search uses 3-fold CV (faster

 per trial, sufficient to rank configs) on the **full dataset** — subsampling

 the Optuna search would risk finding hyperparams that work on the 2019

 pre-ebike distribution but not on the 2021 post-ebike era.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import shap
import statsmodels.api as sm
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

sys.path.insert(0, str(Path().resolve().parent))
from model_training.feature_prep import (
    FEATURE_COLS,
    TARGET_REGRESSION,
    build_preprocessor,
    get_X_y,
    load_training_data,
)


 ## Setup



 Constants in one place so they're easy to adjust. The time estimates assume

 a 32-core machine with LightGBM early stopping active.

 - Ridge Optuna: ~30–60 min (one parameter, fast folds)

 - LightGBM Optuna: ~15–30 min (5 parameters + early stopping; LightGBM on CPU is

   the reason this is far shorter than the XGBoost estimate would have been)

 - Final 5-fold CVs + SHAP: ~20–30 min

 Total: roughly 1.5–2 hours for the full notebook.

In [ ]:
HORIZON_MINUTES = 180
N_SPLITS        = 5       # folds for final CV evaluation of all three models
N_SPLITS_OPT    = 3       # folds during Optuna search — 3 is enough to rank hyperparams
N_JOBS          = -1      # use every core for each model fit
N_TRIALS_RIDGE  = 50      # ~30–60 min
N_TRIALS_LGBM   = 50      # ~15–30 min with early stopping (LightGBM, CPU)
USE_FLOAT32     = True    # halves memory for the linear pipeline

print(f"Cores: {os.cpu_count()}  |  n_jobs={N_JOBS}  |  Ridge trials={N_TRIALS_RIDGE}  |  LGBM trials={N_TRIALS_LGBM}")


 ## Data



 Loading 2019 and 2021 — both years have full availability lags AND trip demand

 data, so all feature families are populated. 2026 stays untouched as the final

 holdout evaluated in 2.04.



 I'm using the full ~12M rows for both Optuna searches. The 2019→2021 shift is

 real: ebike features go from 100% NULL (pre-ebike era) to populated, and the

 network grew from ~1,000 to ~1,500 stations. A subsample could land mostly

 in 2019 and return hyperparameters that are optimal for the pre-ebike

 distribution but suboptimal once ebike features become active. Full data

 means the search sees both eras at their correct proportions.

In [ ]:
df = load_training_data(HORIZON_MINUTES, years=[2019, 2021])
print(f"Loaded: {len(df):,} rows")
print(f"Range:  {df['timestamp'].min()} -> {df['timestamp'].max()}")
print(f"Target: {TARGET_REGRESSION} | range {df[TARGET_REGRESSION].min():.0f}–{df[TARGET_REGRESSION].max():.0f} bikes")


In [ ]:
# Full feature matrix. float32 halves RAM for the linear pipeline (~4 GB vs ~8 GB).
X, y = get_X_y(df, TARGET_REGRESSION)

if USE_FLOAT32:
    num_cols = X.select_dtypes(include=["float64", "float32"]).columns
    X[num_cols] = X[num_cols].astype("float32")
    y = y.astype("float32")

print(f"Feature matrix: {X.shape[0]:,} rows × {X.shape[1]} features  (float32={USE_FLOAT32})")


 ## OLS Baseline: Coefficients, T-Stats, and P-Values



 sklearn's LinearRegression gives predictions and nothing else. I want to

 see which features the model is leaning on, in what direction, and how

 precisely — so I'm fitting statsmodels OLS once on the full training set

 to get the full inference table: coefficients, standard errors, t-stats,

 p-values, and 95% confidence intervals.



 One thing to keep in mind with 12M rows: p-values are almost meaningless

 as a filter here. The t-stat formula is `t = coef / SE`, and SE shrinks

 as `1 / sqrt(n)`. A feature that gives `t = 2` at 1,000 rows gives

 `t = 220` at 12,000,000 rows for the exact same real-world effect size.

 At this scale a p-value is testing whether the effect is *literally zero* —

 and almost nothing in the real world is literally zero. What to focus on

 instead: coefficient magnitude (how many bikes does a 1-unit change actually

 shift?) and confidence interval width (very narrow at 12M rows, so the

 estimate itself is precise). I'm sorting by |t-stat| to surface the features

 the model is most certain about.

In [ ]:
# Preprocess X using the linear pipeline (impute + scale) fitted on all
# training data. For the inference table we fit on the full dataset — this
# is correct for interpretability. The CV metrics below use per-fold fitting.
# Using float32 here to keep the design matrix at ~4 GB instead of ~8 GB.
pre_ols  = build_preprocessor("linear")
X_ols    = pre_ols.fit_transform(X).astype(np.float32)
feat_ols = list(pre_ols.get_feature_names_out())

X_ols_sm = sm.add_constant(X_ols.astype(np.float64))
print(f"OLS design matrix: {X_ols_sm.shape[0]:,} rows × {X_ols_sm.shape[1]} columns (incl. constant)")


In [ ]:
print("Fitting OLS on full training data (may take a few minutes) ...")
ols_model = sm.OLS(y.astype(np.float64), X_ols_sm).fit()
print(f"R²={ols_model.rsquared:.4f}   Adj-R²={ols_model.rsquared_adj:.4f}   F-stat={ols_model.fvalue:.1f}")


In [ ]:
# Coefficient table sorted by |t-stat| so the most influential features surface
# at the top. Remember: at 12M rows every coefficient will have p ≈ 0 — focus
# on coef magnitude and CI width, not the p-value itself.
coef_df = pd.DataFrame({
    "feature":  ["const"] + feat_ols,
    "coef":     ols_model.params,
    "t_stat":   ols_model.tvalues,
    "p_value":  ols_model.pvalues,
    "ci_lower": ols_model.conf_int()[0],
    "ci_upper": ols_model.conf_int()[1],
})
coef_df["abs_t"] = coef_df["t_stat"].abs()
coef_df = coef_df.sort_values("abs_t", ascending=False).reset_index(drop=True)

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_rows", 40)
print("Top 30 features by |t-stat|  (p-values near 0 are expected at n=12M — see note above)")
print(coef_df[["feature","coef","t_stat","p_value","ci_lower","ci_upper"]].head(30).to_string(index=False))


 ## Walk-Forward CV Setup



 TimeSeriesSplit always trains on the past and validates on the future —

 same structure as production. 5 folds for the final evaluation of all three

 models; 3 folds inside each Optuna trial (enough to rank hyperparameter

 configs, faster per trial).

In [ ]:
tscv     = TimeSeriesSplit(n_splits=N_SPLITS)
tscv_opt = TimeSeriesSplit(n_splits=N_SPLITS_OPT)

print("Final CV fold sizes (train, test):")
for i, (tr_idx, te_idx) in enumerate(tscv.split(X), 1):
    print(f"  Fold {i}: {len(tr_idx):>8,} train | {len(te_idx):>7,} test")


 ## Shared CV Utilities



 One loop and one metric function reused for all three models so the

 comparison is consistent.

In [ ]:
def relative_absolute_error(y_true, y_pred):
    """RAE < 1.0 = beats the naive mean-prediction baseline."""
    baseline = np.full_like(y_true, y_true.mean(), dtype=float)
    return np.sum(np.abs(y_pred - y_true)) / np.sum(np.abs(baseline - y_true))


def run_cv(pipeline, X, y, tscv):
    """Return per-fold RAE, validation RMSE, and train RMSE lists.

    Train RMSE is scored on the same fold the model was fitted on, so the
    train-vs-validation gap is an overfitting signal: train RMSE is always
    ≤ validation RMSE, and a *large* gap (train ≪ validation) means the model
    is memorizing the training fold rather than generalizing. The plot below
    visualizes this gap per model.
    """
    fold_raes, fold_rmses, fold_train_rmses = [], [], []
    for train_idx, test_idx in tscv.split(X):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        pipeline.fit(X_tr, y_tr)
        preds_te = pipeline.predict(X_te)
        preds_tr = pipeline.predict(X_tr)
        fold_raes.append(relative_absolute_error(y_te.values, preds_te))
        fold_rmses.append(np.sqrt(mean_squared_error(y_te, preds_te)))
        fold_train_rmses.append(np.sqrt(mean_squared_error(y_tr, preds_tr)))
    return fold_raes, fold_rmses, fold_train_rmses


 ## Model 1: Linear Regression (Floor Benchmark)



 No tuning — this is the floor every other model has to beat. The statsmodels

 OLS above is this same model; the sklearn version here gives us RMSE/RAE

 inside the walk-forward CV loop so the metrics are comparable to Ridge and

 LightGBM.

In [ ]:
print("CV: Linear Regression ...", end=" ", flush=True)
lr_pipeline = Pipeline([
    ("pre",   build_preprocessor("linear")),
    ("model", LinearRegression()),
])
lr_raes, lr_rmses, lr_train_rmses = run_cv(lr_pipeline, X, y, tscv)
results = {"Linear Regression": {"rae": lr_raes, "rmse": lr_rmses, "train_rmse": lr_train_rmses}}
print(f"done — RAE {np.mean(lr_raes):.3f} ± {np.std(lr_raes):.3f}  RMSE {np.mean(lr_rmses):.3f} ± {np.std(lr_rmses):.3f}")


 ## Model 2: Ridge — Optuna Hyperparameter Search



 Ridge adds L2 regularization controlled by alpha. The previous untuned run

 used alpha=1.0 and matched Linear exactly — meaning 1.0 is too weak to change

 anything on this dataset. Optuna searches alpha log-uniformly across six

 orders of magnitude (1e-4 to 1e4) to find a value that actually regularizes.



 Each trial runs 3-fold walk-forward CV. The pipeline fits the preprocessor

 (imputer + scaler) fresh inside each fold, so no test statistics leak into

 the imputation or scaling.

In [ ]:
# Pre-compute preprocessed fold arrays for Ridge Optuna. Only alpha changes
# between trials — the imputer and scaler output is identical every time.
# Fitting the preprocessor 3 times (once per fold) instead of 150 times
# (50 trials × 3 folds) drops Ridge Optuna from ~45 min to ~5 min.
print("Pre-computing Ridge fold transforms (3 fits instead of 150) ...")
_ridge_folds = []
for train_idx, test_idx in tscv_opt.split(X):
    pre      = build_preprocessor("linear")
    X_tr_pre = pre.fit_transform(X.iloc[train_idx])
    X_te_pre = pre.transform(X.iloc[test_idx])
    _ridge_folds.append((
        X_tr_pre,
        X_te_pre,
        y.iloc[train_idx].values,
        y.iloc[test_idx].values,
    ))
print("Done.")


def objective_ridge(trial):
    alpha = trial.suggest_float("alpha", 1e-4, 1e4, log=True)
    rmses = []
    for X_tr_pre, X_te_pre, y_tr, y_te in _ridge_folds:
        model = Ridge(alpha=alpha)
        model.fit(X_tr_pre, y_tr)
        rmses.append(np.sqrt(mean_squared_error(y_te, model.predict(X_te_pre))))
    return np.mean(rmses)


In [ ]:
# n_jobs=8 runs 8 Ridge trials simultaneously. Each Ridge fit uses ~2 BLAS
# threads (OpenBLAS compile cap), so 8 parallel trials = 16 threads — well
# within the 32-core budget. Trials are independent so threading is safe.
print(f"Ridge Optuna: {N_TRIALS_RIDGE} trials × {N_SPLITS_OPT}-fold CV on full {len(X):,} rows ...")
ridge_study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
ridge_study.optimize(objective_ridge, n_trials=N_TRIALS_RIDGE, n_jobs=8, show_progress_bar=True)

best_ridge_alpha = ridge_study.best_params["alpha"]
print(f"Best alpha: {best_ridge_alpha:.6f}   best-trial RMSE: {ridge_study.best_value:.4f}")


In [ ]:
# Final evaluation: 5-fold CV with the tuned alpha. The preprocessing pipeline
# is rebuilt and refitted inside each fold — same process as Linear Regression.
print("Ridge final 5-fold CV ...", end=" ", flush=True)
ridge_pipeline = Pipeline([
    ("pre",   build_preprocessor("linear")),
    ("model", Ridge(alpha=best_ridge_alpha)),
])
ridge_raes, ridge_rmses, ridge_train_rmses = run_cv(ridge_pipeline, X, y, tscv)
results["Ridge (tuned)"] = {"rae": ridge_raes, "rmse": ridge_rmses, "train_rmse": ridge_train_rmses}
print(f"done — RAE {np.mean(ridge_raes):.3f} ± {np.std(ridge_raes):.3f}  RMSE {np.mean(ridge_rmses):.3f} ± {np.std(ridge_rmses):.3f}")


 ## Model 3: LightGBM — Optuna Hyperparameter Search



 LightGBM is the expected winner — the untuned XGBoost run (RMSE 5.053) already

 beat Ridge by 13%, and LightGBM lands in the same accuracy ballpark on tabular

 data while training far faster on CPU. The question is how much tuned

 hyperparameters can improve on the baseline.



 One thing worth flagging up front: LightGBM grows trees **leaf-wise** (it always

 splits the leaf with the largest loss reduction), whereas XGBoost grows

 **level-wise**. Leaf-wise converges faster but can overfit if left unconstrained,

 which is exactly why `num_leaves` and `min_child_samples` are in the search — they

 are the guardrails. The five parameters, and their XGBoost analogues:

 - `learning_rate` — same knob as XGBoost.

 - `num_leaves` — LightGBM's complexity control instead of `max_depth`. It sets the

   leaf count directly rather than a depth cap, so I search it on a log scale.

 - `feature_fraction` — column subsampling per tree (XGBoost's `colsample_bytree`).

 - `bagging_fraction` + `bagging_freq=1` — row subsampling per iteration

   (XGBoost's `subsample`; LightGBM needs `bagging_freq` set for bagging to engage).

 - `min_child_samples` — minimum rows in a leaf (XGBoost's `min_child_weight`),

   the main leaf-wise overfitting guard.



 `n_estimators` is handled by early stopping inside each trial: LightGBM stops

 adding trees when validation RMSE hasn't improved for 50 consecutive rounds.

 This keeps bad configs short and lets good ones run as long as they need to.

 The average stopping point across folds is stored and used for the final clean

 evaluation (where early stopping is off so the test fold isn't used to decide

 tree count).



 The preprocessor (one-hot encoding only, no imputer or scaler) is fitted once on

 the full dataset before the search. I'm keeping the OHE rather than using

 LightGBM's native categorical handling so the same `feature_prep` path serves all

 three models. Categories are fixed — season has 4 values, station_role has 3 +

 unknown — so fitting once vs 150 times (50 trials × 3 folds) gives identical

 transformations and saves time.

In [ ]:
# Pre-fit the OHE preprocessor once. Categories are fixed so this is equivalent
# to fitting per-fold and avoids 150 redundant fits. LightGBM and XGBoost share
# the same tree path in feature_prep, so "lightgbm" here == OHE only.
_pre_lgbm = build_preprocessor("lightgbm").fit(X)


def objective_lgbm(trial):
    params = dict(
        learning_rate     = trial.suggest_float("learning_rate",     0.01, 0.3,  log=True),
        num_leaves        = trial.suggest_int(  "num_leaves",        15,   255,  log=True),
        feature_fraction  = trial.suggest_float("feature_fraction",  0.5,  1.0),
        bagging_fraction  = trial.suggest_float("bagging_fraction",  0.5,  1.0),
        min_child_samples = trial.suggest_int(  "min_child_samples", 5,    100),
        bagging_freq      = 1,      # required for bagging_fraction to take effect
        n_estimators      = 1000,
        n_jobs            = N_JOBS,
        random_state      = 42,
        verbose           = -1,
    )
    rmses, n_trees = [], []
    for train_idx, test_idx in tscv_opt.split(X):
        X_tr = _pre_lgbm.transform(X.iloc[train_idx])
        X_te = _pre_lgbm.transform(X.iloc[test_idx])
        y_tr = y.iloc[train_idx]
        y_te = y.iloc[test_idx]

        model = LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_te, y_te)],
            eval_metric="rmse",
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )

        rmses.append(np.sqrt(mean_squared_error(y_te, model.predict(X_te))))
        # best_iteration_ is the LightGBM analogue of XGBoost's best_iteration.
        n_trees.append(model.best_iteration_)

    trial.set_user_attr("n_estimators", int(np.mean(n_trees)))
    return np.mean(rmses)


In [ ]:
print(f"LightGBM Optuna: {N_TRIALS_LGBM} trials × {N_SPLITS_OPT}-fold CV + early stopping on full {len(X):,} rows ...")
lgbm_study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
lgbm_study.optimize(objective_lgbm, n_trials=N_TRIALS_LGBM, show_progress_bar=True)

best_lgbm_params = lgbm_study.best_params.copy()
best_lgbm_n_est  = lgbm_study.best_trial.user_attrs["n_estimators"]
print(f"Best params:      {best_lgbm_params}")
print(f"Best n_estimators (avg early-stop across folds): {best_lgbm_n_est}")
print(f"Best trial RMSE:  {lgbm_study.best_value:.4f}")


 ### LightGBM learning curve



 Before the final evaluation I want to *see* how the tuned model trains. I fit

 one LightGBM with the best params on the last (largest-train) walk-forward fold,

 recording train and validation RMSE at every boosting round. This is the classic

 learning curve: train RMSE always keeps falling as trees are added, but the

 validation curve is what matters — once it flattens (or turns up) while train

 keeps dropping, the model has started memorizing and extra trees just overfit.

 The dashed line marks where early stopping halted, which should sit right around

 the elbow of the validation curve. The gap between the two curves at that point

 is the same overfitting signal the per-fold bar chart shows, viewed round-by-round.

In [ ]:
*_, (lc_tr_idx, lc_te_idx) = list(tscv.split(X))   # last fold = largest train set
X_lc_tr = _pre_lgbm.transform(X.iloc[lc_tr_idx])
X_lc_te = _pre_lgbm.transform(X.iloc[lc_te_idx])
y_lc_tr = y.iloc[lc_tr_idx]
y_lc_te = y.iloc[lc_te_idx]

lc_evals = {}
lc_model = LGBMRegressor(
    **best_lgbm_params,
    bagging_freq = 1,
    n_estimators = 1000,
    n_jobs       = N_JOBS,
    random_state = 42,
    verbose      = -1,
)
lc_model.fit(
    X_lc_tr, y_lc_tr,
    eval_set   = [(X_lc_tr, y_lc_tr), (X_lc_te, y_lc_te)],
    eval_names = ["train", "validation"],
    eval_metric= "rmse",
    callbacks  = [
        lgb.early_stopping(50, verbose=False),
        lgb.record_evaluation(lc_evals),
    ],
)

train_curve = lc_evals["train"]["rmse"]
val_curve   = lc_evals["validation"]["rmse"]

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_curve) + 1), train_curve, label="train RMSE")
plt.plot(range(1, len(val_curve) + 1),   val_curve,   label="validation RMSE")
plt.axvline(lc_model.best_iteration_, ls="--", color="gray",
            label=f"early stop @ {lc_model.best_iteration_} rounds")
plt.xlabel("boosting round")
plt.ylabel("RMSE")
plt.title("LightGBM Learning Curve — 3hr Horizon (last walk-forward fold)")
plt.legend()
plt.tight_layout()
Path("../reports/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("../reports/figures/2.02_lgbm_learning_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to reports/figures/2.02_lgbm_learning_curve.png  "
      f"(final train RMSE {train_curve[lc_model.best_iteration_ - 1]:.3f} "
      f"vs validation {val_curve[lc_model.best_iteration_ - 1]:.3f})")


In [ ]:
# Final 5-fold CV with fixed n_estimators — early stopping is off so the test
# fold is not used to decide tree count. This keeps the evaluation metrics clean.
print("LightGBM final 5-fold CV ...", end=" ", flush=True)
lgbm_final_pipeline = Pipeline([
    ("pre",   build_preprocessor("lightgbm")),
    ("model", LGBMRegressor(
        **best_lgbm_params,
        bagging_freq = 1,
        n_estimators = best_lgbm_n_est,
        n_jobs       = N_JOBS,
        random_state = 42,
        verbose      = -1,
    )),
])
lgbm_raes, lgbm_rmses, lgbm_train_rmses = run_cv(lgbm_final_pipeline, X, y, tscv)
results["LGBMRegressor (tuned)"] = {"rae": lgbm_raes, "rmse": lgbm_rmses, "train_rmse": lgbm_train_rmses}
print(f"done — RAE {np.mean(lgbm_raes):.3f} ± {np.std(lgbm_raes):.3f}  RMSE {np.mean(lgbm_rmses):.3f} ± {np.std(lgbm_rmses):.3f}")


 ## Results



 Each cell shows mean ± std across folds. The std matters as much as the mean —

 a model with a low mean but high std is less trustworthy across different time

 windows than one that's consistently good.

In [ ]:
rows = []
for name, vals in results.items():
    rows.append({
        "Model":              name,
        "Train RMSE (m ± s)": f"{np.mean(vals['train_rmse']):.3f} ± {np.std(vals['train_rmse']):.3f}",
        "Val RMSE (m ± s)":   f"{np.mean(vals['rmse']):.3f} ± {np.std(vals['rmse']):.3f}",
        "RAE  (m ± s)":       f"{np.mean(vals['rae']):.3f} ± {np.std(vals['rae']):.3f}",
        "Overfit gap":        f"{np.mean(vals['rmse']) - np.mean(vals['train_rmse']):.3f}",
    })

metrics_df = pd.DataFrame(rows).set_index("Model")
print(metrics_df.to_string())
print("\nRAE < 1.0 = beats the naive mean-prediction baseline.")
print("Overfit gap = mean val RMSE − mean train RMSE; larger = more overfitting.")


 ### Per-fold detail



 The summary above is the headline. The fold-by-fold breakdown shows whether

 any model is especially sensitive to a particular time window.

In [ ]:
for name, vals in results.items():
    print(f"\n{name}")
    for i, (rae, rmse, tr_rmse) in enumerate(zip(vals["rae"], vals["rmse"], vals["train_rmse"]), 1):
        print(f"  Fold {i}: train RMSE={tr_rmse:.3f}  val RMSE={rmse:.3f}  RAE={rae:.3f}")


 ### Train vs Validation RMSE by Fold (Overfitting Check)



 This is the visual version of the overfit-gap column above. For each model I plot

 train RMSE next to validation RMSE for every walk-forward fold. Two things to read:

 - **Gap within a fold** — train bar much shorter than the validation bar means the

   model fits the training window far better than the held-out future, i.e. it's

   overfitting. I expect Linear/Ridge to show almost no gap (they can't overfit much

   with this many rows) and LightGBM to show some gap — the question is how much.

 - **Trend across folds** — TimeSeriesSplit grows the training set each fold, so a

   model that's generalizing should see the validation bars hold steady or improve

   left-to-right rather than swing around.

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4), sharey=True)
if len(results) == 1:
    axes = [axes]
for ax, (name, vals) in zip(axes, results.items()):
    folds = np.arange(1, len(vals["rmse"]) + 1)
    width = 0.38
    ax.bar(folds - width / 2, vals["train_rmse"], width, label="train")
    ax.bar(folds + width / 2, vals["rmse"],       width, label="validation")
    ax.set_title(name)
    ax.set_xlabel("walk-forward fold")
    ax.set_xticks(folds)
axes[0].set_ylabel("RMSE")
axes[0].legend()
fig.suptitle("Train vs Validation RMSE by Walk-Forward Fold — 3hr Horizon", y=1.02)
plt.tight_layout()
Path("../reports/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("../reports/figures/2.02_train_vs_val_rmse.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to reports/figures/2.02_train_vs_val_rmse.png")


 ## LightGBM: SHAP Feature Importance



 I'm fitting LightGBM on the full training data with the tuned hyperparameters,

 then running SHAP on a 100k subsample. Built-in LightGBM importance (split count

 or gain) can mislead — a feature that appears in many splits looks important

 even if each split contributes little to the prediction. SHAP measures the

 actual contribution of each feature to each prediction, averaged over the

 sample, which gives a more honest picture of what the model relies on.



 SHAP is the one place subsampling is fine — TreeExplainer is O(rows) and the

 mean |SHAP| ranking is stable at 100k rows. TreeExplainer supports LightGBM

 natively, same as XGBoost.

In [ ]:
print("Fitting tuned LightGBM on full training data for SHAP ...")
lgbm_final_pipeline.fit(X, y)
print("Done.")


In [ ]:
rng      = np.random.default_rng(42)
shap_idx = np.sort(rng.choice(len(X), size=min(100_000, len(X)), replace=False))
X_shap   = X.iloc[shap_idx].reset_index(drop=True)

pre_shap   = lgbm_final_pipeline.named_steps["pre"]
X_shap_tr  = pre_shap.transform(X_shap)
feat_names = list(pre_shap.get_feature_names_out())
print(f"SHAP sample: {len(X_shap):,} rows × {len(feat_names)} features")


In [ ]:
print("Computing SHAP values ...")
explainer   = shap.TreeExplainer(lgbm_final_pipeline.named_steps["model"])
shap_values = explainer.shap_values(X_shap_tr)
print("Done.")


In [ ]:
Path("../reports/figures").mkdir(parents=True, exist_ok=True)

shap.summary_plot(
    shap_values,
    X_shap_tr,
    feature_names=feat_names,
    plot_type="bar",
    max_display=20,
    show=False,
)
plt.title("LightGBM Feature Importance (mean |SHAP|) — 3hr Horizon (tuned)", pad=12)
plt.tight_layout()
plt.savefig("../reports/figures/2.02_shap_importance_3hr.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to reports/figures/2.02_shap_importance_3hr.png")


In [ ]:
# ## Conclusion
#

# LightGBM wins by a mile. Val RMSE 4.92 vs 5.83 for linear — that's about 1 fewer bike
# of error on average at the 3-hour horizon, which actually matters when a station only has
# 8 bikes to begin with. RAE 0.309 means it's cutting error by 69% compared to just
# guessing the station's average every time.

# Ridge with alpha=4,315 is basically identical to plain linear regression. Not surprising —
# at 12 million rows the data already pins down the coefficients so tightly that
# regularization has nothing left to do.

# The overfit gap on LightGBM (train 3.29, val 4.92) is something I want to keep an eye on.
# Folds 3–5 are consistent around 5.0, but fold 2 jumps to 5.54 — that's the 2019→2021 
# transition, where ebikes rolled out and the network expanded significantly. The model
# trained on 2019 data is seeing a pretty different world in 2021. The learning curve looks
# fine though — converges cleanly around 469 trees with no sign of runaway overfitting.
# The real test will be the 2026 holdout in 2.04.

# SHAP is unsurprising for a 3-hour horizon: current availability and recent net flow
# dominate, which makes sense — at 3 hours out the station's current state still tells you
# a lot. Season and time features matter but are secondary. I'll do the full beeswarm and
# dependence plots in 2.06 once all horizons are trained and there's more to say.

# Carrying forward into 2.04: learning_rate=0.117, num_leaves=199, feature_fraction=0.85,
# bagging_fraction=0.73, min_child_samples=54, n_estimators=469.
